In [1]:
!pip install -q timm psutil

In [1]:
import os
import time
import psutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader, Subset
import timm

# 1. Fixed Seed for Identical Initialization & Shuffling
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# 2. Auto-detect Device (TPU -> GPU -> CPU)
USE_XLA = False
DEVICE_LABEL = "CPU"
device = torch.device("cpu")

try:
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
    USE_XLA = True
    DEVICE_LABEL = "TPU"
except Exception:
    try:
        tpu_dev = torch.device("tpu")
        _ = torch.zeros(1, device=tpu_dev)
        device = tpu_dev
        DEVICE_LABEL = "TPU"
    except Exception:
        if torch.cuda.is_available():
            device = torch.device("cuda")
            DEVICE_LABEL = "GPU"
        else:
            device = torch.device("cpu")
            DEVICE_LABEL = "CPU"

print(f"Active Training Device: {DEVICE_LABEL} ({device})")

def sync_device():
    if DEVICE_LABEL == "GPU":
        torch.cuda.synchronize()
    elif DEVICE_LABEL == "TPU" and USE_XLA:
        xm.mark_step()
        xm.wait_device_ops()

# 3. Load Fixed 2,048-Image Subset of CIFAR-10 (64 steps/epoch at BS=32)
BATCH_SIZE = 32
EPOCHS = 3
NUM_SAMPLES = 2048

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

full_train = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform
)

g = torch.Generator().manual_seed(SEED)
subset_indices = torch.randperm(len(full_train), generator=g)[:NUM_SAMPLES].tolist()
train_subset = Subset(full_train, subset_indices)

train_loader = DataLoader(
    train_subset, batch_size=BATCH_SIZE, shuffle=True,
    drop_last=True, generator=torch.Generator().manual_seed(SEED)
)

# 4. Model, Loss, Optimizer (Identical Across All Devices)
model = timm.create_model("mobilevit_s", pretrained=True, num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

os.makedirs("checkpoints", exist_ok=True)
if DEVICE_LABEL == "GPU":
    torch.cuda.reset_peak_memory_stats()

# 5. Training Loop
rows = []
for epoch in range(1, EPOCHS + 1):
    model.train()
    step_times_ms = []
    running_loss = 0.0
    correct = 0
    total = 0

    epoch_t0 = time.perf_counter()

    for step, (imgs, labels) in enumerate(train_loader, start=1):
        t0 = time.perf_counter()

        imgs = imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()

        if DEVICE_LABEL == "TPU" and USE_XLA:
            xm.optimizer_step(optimizer)
        else:
            optimizer.step()

        # Forces device completion on TPU/GPU and extracts scalar metrics
        loss_val = loss.item()
        sync_device()

        t1 = time.perf_counter()
        step_times_ms.append((t1 - t0) * 1000.0)

        preds = outputs.detach().argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        running_loss += loss_val

    epoch_time_s = time.perf_counter() - epoch_t0

    # Peak memory tracking
    if DEVICE_LABEL == "GPU":
        peak_mem_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
    else:
        peak_mem_mb = psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2)

    # Save checkpoint after every epoch (Free Colab disconnect protection)
    ckpt_path = f"checkpoints/mobilevit_{DEVICE_LABEL.lower()}_epoch{epoch}.pt"
    torch.save({
        "epoch": epoch,
        "model_state_dict": {k: v.cpu() for k, v in model.state_dict().items()},
        "loss": running_loss / len(train_loader),
    }, ckpt_path)

    # Separate first step (includes TPU graph compile in Epoch 1) from steady steps
    first_step_ms = step_times_ms[0]
    steady_steps = step_times_ms[1:] if (epoch == 1 and len(step_times_ms) > 1) else step_times_ms

    med_step_ms = float(np.median(steady_steps))
    p95_step_ms = float(np.percentile(steady_steps, 95))
    avg_loss = running_loss / len(train_loader)
    acc_pct = 100.0 * correct / total

    rows.append({
        "Device": DEVICE_LABEL,
        "Model": "MobileViT-S",
        "Epoch": epoch,
        "Steps": len(train_loader),
        "Epoch_Time_s": round(epoch_time_s, 2),
        "First_Step_ms": round(first_step_ms, 2),
        "Steady_Median_Step_ms": round(med_step_ms, 2),
        "Steady_P95_Step_ms": round(p95_step_ms, 2),
        "Train_Loss": round(avg_loss, 4),
        "Train_Acc_%": round(acc_pct, 2),
        "Peak_Mem_MB": round(peak_mem_mb, 2),
    })

    print(
        f"[{DEVICE_LABEL}] Epoch {epoch}/{EPOCHS} | "
        f"Time: {epoch_time_s:6.2f}s | 1st Step: {first_step_ms:8.1f} ms | "
        f"Steady Step: {med_step_ms:6.2f} ms | Loss: {avg_loss:.4f} | "
        f"Acc: {acc_pct:5.2f}% | Mem: {peak_mem_mb:7.1f} MB"
    )

out_csv = f"training_{DEVICE_LABEL.lower()}.csv"
train_df = pd.DataFrame(rows)
train_df.to_csv(out_csv, index=False)
print(f"\nSaved {out_csv}")

Active Training Device: CPU (cpu)


100%|██████████| 170M/170M [00:04<00:00, 41.5MB/s]


model.safetensors: reconstructing file:   0%|          |  0.00B / 22.4MB            

model.safetensors: downloading bytes:           |  0.00B            

[CPU] Epoch 1/3 | Time: 310.14s | 1st Step:   5962.3 ms | Steady Step: 4662.09 ms | Loss: 1.1581 | Acc: 66.65% | Mem:  1478.0 MB
[CPU] Epoch 2/3 | Time: 304.86s | 1st Step:   4783.1 ms | Steady Step: 4643.32 ms | Loss: 0.4908 | Acc: 85.01% | Mem:  1475.3 MB
[CPU] Epoch 3/3 | Time: 312.87s | 1st Step:   4441.1 ms | Steady Step: 4732.34 ms | Loss: 0.3267 | Acc: 89.55% | Mem:  1456.9 MB

Saved training_cpu.csv
